# Unidad 21. Combinación de datos, fechas y exportación

**Duración orientativa:** 7 horas · **UT3**

Los datos relevantes suelen estar repartidos entre varias tablas. Esta unidad enseña a relacionarlas mediante claves, comprobar el resultado de una unión, interpretar fechas y guardar resultados reproducibles.

## Resultados de aprendizaje

Al finalizar podrás concatenar tablas compatibles, unirlas con merge, distinguir inner y left join, detectar claves sin correspondencia, convertir fechas con control de errores y exportar resultados sin incluir el índice técnico.

**Material:** datos/pandas/21_estaciones.csv y datos/pandas/21_mantenimiento.csv.

## 1. Claves y relaciones

Una clave identifica una entidad. estacion_id identifica una estación; puede aparecer una vez en una tabla de estaciones y varias veces en una tabla de mantenimientos. Esta es una relación uno a muchos. Antes de unir, comprueba el significado, tipo y unicidad de las claves. Un merge incorrecto puede multiplicar filas y producir resultados aparentemente plausibles pero falsos.

In [ ]:
from pathlib import Path
import pandas as pd

def ruta(nombre):
    candidata = Path('..') / 'datos' / 'pandas' / nombre
    return candidata if candidata.exists() else Path('datos') / 'pandas' / nombre

estaciones = pd.read_csv(ruta('21_estaciones.csv'))
mantenimiento = pd.read_csv(ruta('21_mantenimiento.csv'))
display(estaciones)
display(mantenimiento)

## 2. Concatenar y unir

concat apila o coloca tablas que ya tienen una estructura compatible. merge relaciona columnas mediante una clave, como una unión SQL. Un inner join conserva solo claves presentes en ambas tablas. Un left join conserva todas las filas de la tabla situada a la izquierda y muestra ausencias cuando no encuentra correspondencia.

Antes de ejecutar, predice: EST-04 no tiene mantenimiento; EST-99 tiene mantenimiento pero no existe como estación. Esa predicción es una comprobación conceptual, no un detalle opcional.

In [ ]:
inner = estaciones.merge(mantenimiento, on='estacion_id', how='inner', validate='one_to_many')
left = estaciones.merge(mantenimiento, on='estacion_id', how='left', validate='one_to_many', indicator=True)
print('Filas inner:', len(inner))
display(inner)
display(left[['estacion_id', 'zona', 'orden_id', '_merge']])

validate='one_to_many' convierte una suposición en una comprobación: cada estación puede aparecer una vez en la tabla izquierda y muchas veces a la derecha. indicator=True crea _merge, útil para localizar filas solo presentes en la izquierda o en ambas. Las claves de la derecha sin pareja se detectan invirtiendo la unión o usando un outer join.

In [ ]:
auditoria = estaciones.merge(mantenimiento, on='estacion_id', how='outer', indicator=True)
auditoria.loc[auditoria['_merge'] != 'both', ['estacion_id', 'zona', 'orden_id', '_merge']]

## 3. Fechas

Un CSV guarda texto. pd.to_datetime interpreta ese texto como fecha y hora. errors='coerce' transforma formatos inválidos en NaT, una ausencia temporal visible que se debe revisar. Tras la conversión, el accesorio dt permite obtener año, mes o día de la semana. No confundas una fecha de alta con una fecha de mantenimiento: cada variable responde a una pregunta temporal distinta.

In [ ]:
mantenimiento_fechas = mantenimiento.copy()
mantenimiento_fechas['fecha_mantenimiento'] = pd.to_datetime(mantenimiento_fechas['fecha_mantenimiento'], errors='coerce')
mantenimiento_fechas['anio'] = mantenimiento_fechas['fecha_mantenimiento'].dt.year
mantenimiento_fechas['mes'] = mantenimiento_fechas['fecha_mantenimiento'].dt.month
mantenimiento_fechas['dia_semana'] = mantenimiento_fechas['fecha_mantenimiento'].dt.day_name()
mantenimiento_fechas

## 4. Exportar resultados

CSV facilita interoperabilidad y es adecuado para tablas simples. JSON conserva estructuras con más flexibilidad. Excel puede ser útil para intercambio con usuarios, pero añade una dependencia opcional. Al exportar un DataFrame cuyo índice no es una variable del análisis, usa index=False. Antes de guardar, revisa ruta, nombres de columnas, unidades y si el archivo contiene información que no deba compartirse.

In [37]:
salida = Path('salidas')
salida.mkdir(exist_ok=True)
ruta_salida = salida / '21_mantenimiento_con_zona.csv'
inner.to_csv(ruta_salida, index=False, encoding='utf-8')
print('Archivo creado:', ruta_salida.resolve())
print('¿Existe?', ruta_salida.exists())

Archivo creado: C:\Users\pajim\OneDrive\Escritorio\Master\Programacion IA\Proyecto\unidades\salidas\21_mantenimiento_con_zona.csv
¿Existe? True


## Errores frecuentes

- Unir por una columna con el mismo nombre pero distinto significado.
- No comprobar claves repetidas antes de merge.
- Usar inner join y perder filas sin advertirlo.
- Convertir fechas sin revisar NaT.
- Exportar el índice como una columna accidental.
- Tratar un archivo exportado como sustituto de documentar el proceso que lo produjo.

## Ejercicios

**Ejercicio 21.01.** Comprueba que estacion_id es único en estaciones y cuenta cuántas órdenes tiene cada estación en mantenimiento.

**Ejercicio 21.02.** Realiza un left join de estaciones con mantenimiento y localiza la estación sin órdenes.

**Ejercicio 21.03.** Usa un outer join para localizar la orden asociada a una estación desconocida. Explica por qué no debes eliminarla sin investigarla.

**Ejercicio 21.04.** Convierte fecha_alta y fecha_mantenimiento a tipo temporal. Extrae el mes de ambas y comprueba ausencias.

**Ejercicio 21.05.** Exporta el inner join a salidas/21_resumen_mantenimiento.csv con index=False y vuelve a leerlo para comprobar su forma.

In [ ]:
# Ejercicio 21.01. Comprueba que estacion_id es único en estaciones y cuenta cuántas órdenes tiene cada estación en mantenimiento.
estacion_unica = estaciones['estacion_id'].is_unique
print(f"Estacion es unica en estaciones?: {estacion_unica}")
ordenes_por_estacion=mantenimiento.groupby('estacion_id').size()
ordenes_por_estacion


In [ ]:
# Ejercicio 21.02. Realiza un left join de estaciones con mantenimiento y localiza la estación sin órdenes.
left = estaciones.merge(mantenimiento, on='estacion_id', how='left', validate='one_to_many')
left
# La estacion 4 no tiene ordenes

In [ ]:
mantenimiento_fechas = mantenimiento.copy()
mantenimiento_fechas['fecha_mantenimiento'] = pd.to_datetime(mantenimiento_fechas['fecha_mantenimiento'], errors='coerce')
mantenimiento_fechas['anio'] = mantenimiento_fechas['fecha_mantenimiento'].dt.year
mantenimiento_fechas['mes'] = mantenimiento_fechas['fecha_mantenimiento'].dt.month
mantenimiento_fechas['dia_semana'] = mantenimiento_fechas['fecha_mantenimiento'].dt.day_name()
mantenimiento_fechas

In [ ]:
# Ejercicio 21.03. Usa un outer join para localizar la orden asociada a una estación desconocida. Explica por qué no debes eliminarla sin investigarla.
outer = estaciones.merge(mantenimiento, on='estacion_id', how='outer', validate='one_to_many')
outer
# La orden asociada es M-005. Esta orden puede ser correcta, pero la clave estacion_id puede estar mal escrita

In [ ]:
# Ejercicio 21.04. Convierte fecha_alta y fecha_mantenimiento a tipo temporal. Extrae el mes de ambas y comprueba ausencias.
estaciones_fechas = estaciones.copy()
mantenimiento_fechas = mantenimiento.copy()
estaciones_fechas['fecha_alta'] = pd.to_datetime(estaciones_fechas['fecha_alta'], errors='coerce')
mantenimiento_fechas['fecha_mantenimiento'] = pd.to_datetime(mantenimiento_fechas['fecha_mantenimiento'], errors = "coerce")

mantenimiento_fechas['mes'] = mantenimiento_fechas['fecha_mantenimiento'].dt.month
estaciones_fechas['mes'] = estaciones_fechas['fecha_alta'].dt.month

# Comprobacion de ausencias
print(estaciones_fechas[['fecha_alta', 'mes']].isna().sum())
print(mantenimiento_fechas[['fecha_mantenimiento', 'mes']].isna().sum())

In [ ]:
salida = Path('salidas')
salida.mkdir(exist_ok=True)
ruta_salida = salida / '21_mantenimiento_con_zona.csv'
inner.to_csv(ruta_salida, index=False, encoding='utf-8')
print('Archivo creado:', ruta_salida.resolve())
print('¿Existe?', ruta_salida.exists())

In [41]:
# Ejercicio 21.05. Exporta el inner join a salidas/21_resumen_mantenimiento.csv con index=False y vuelve a leerlo para comprobar su forma.
inner = estaciones.merge(mantenimiento, on='estacion_id', how='inner', validate='one_to_many')
ruta_salida = salida / '21_resumen_mantenimiento.csv'
inner.to_csv(ruta_salida, index=False, encoding='utf-8')
comprobacion = pd.read_csv(ruta_salida)
comprobacion.shape

(4, 8)

## Autoevaluación

1. ¿Qué representa una clave y qué significa una relación uno a muchos?
2. ¿Qué filas conserva un inner join? ¿Y un left join?
3. ¿Por qué validate es una práctica recomendable?
4. ¿Qué representa NaT?
5. ¿Por qué index=False suele ser adecuado al exportar?

En la unidad 22 usarás tablas preparadas para comunicar resultados mediante gráficos.